<div style="display: flex; justify-content: space-between; align-items: center;">
    <div style="text-align: left; flex: 4">
        <strong>Author:</strong> Amirhossein Heydari — 
        📧 <a href="mailto:amirhosseinheydari78@gmail.com">amirhosseinheydari78@gmail.com</a> — 
        🐙 <a href="https://github.com/mr-pylin/pandas-workshop" target="_blank" rel="noopener">github.com/mr-pylin</a>
    </div>
    <div style="text-align: right; flex: 1;">
        <a href="https://pandas.pydata.org/" target="_blank" rel="noopener noreferrer">
            <img src="../assets/images/pandas/logo/pandas_white.svg" 
                 alt="Pandas Logo"
                 style="max-height: 48px; width: auto; background-color: #1f1f1f; border-radius: 8px;">
        </a>
    </div>
</div>
<hr>


**Table of contents**<a id='toc0_'></a>    
- [Dependencies](#toc1_)    
- [Load Datasets](#toc2_)    
  - [Titanic](#toc2_1_)    
  - [Sales](#toc2_2_)    
- [Merging, Joining, and Reshaping](#toc3_)    
  - [Introduction to Data Combination](#toc3_1_)    
    - [Why merging and reshaping matter](#toc3_1_1_)    
    - [Comparison of concatenation, merging, and reshaping](#toc3_1_2_)    
    - [Overview of relational data concepts](#toc3_1_3_)    
  - [Concatenation and Appending](#toc3_2_)    
    - [Using `pd.concat()` for stacking DataFrames](#toc3_2_1_)    
    - [Ignoring indexes and adding keys](#toc3_2_2_)    
  - [Merging and Joining](#toc3_3_)    
    - [The `merge()` function and SQL-style joins](#toc3_3_1_)    
      - [Inner Join](#toc3_3_1_1_)    
      - [Outer join](#toc3_3_1_2_)    
      - [Left join](#toc3_3_1_3_)    
      - [Right join](#toc3_3_1_4_)    
    - [Joining on indexes and multi-indexes](#toc3_3_2_)    
    - [Handling overlapping column names (`suffixes` parameter)](#toc3_3_3_)    
  - [Pivoting and Reshaping Data](#toc3_4_)    
    - [Converting long to wide format with `pivot()`](#toc3_4_1_)    
    - [Aggregating with `pivot_table()`](#toc3_4_2_)    
  - [Melting and Long-Form Transformations](#toc3_5_)    
    - [Using `melt()` to unpivot data](#toc3_5_1_)    
    - [Reversing a melt operation](#toc3_5_2_)    
  - [Stacking and Unstacking](#toc3_6_)    
    - [Using `stack()` and `unstack()` to reshape multi-indexed data](#toc3_6_1_)    
      - [Stack](#toc3_6_1_1_)    
      - [Unstack](#toc3_6_1_2_)    
    - [Combining reshaping with aggregation for advanced layouts](#toc3_6_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Dependencies](#toc0_)


In [ ]:
import pandas as pd

In [ ]:
# disable wrapping entirely
pd.set_option("display.expand_frame_repr", False)

# <a id='toc2_'></a>[Load Datasets](#toc0_)


## <a id='toc2_1_'></a>[Titanic](#toc0_)


In [ ]:
TITANIC_TRAIN_PATH = (
    r"https://raw.githubusercontent.com/mr-pylin/datasets/refs/heads/main/data/tabular-data/titanic/train.csv"
)
TITANIC_TEST_PATH = (
    r"https://raw.githubusercontent.com/mr-pylin/datasets/refs/heads/main/data/tabular-data/titanic/test.csv"
)

In [ ]:
titanic_train_df = pd.read_csv(TITANIC_TRAIN_PATH, encoding="utf-8")

# log
titanic_train_df.head()

In [ ]:
titanic_test_df = pd.read_csv(TITANIC_TEST_PATH, encoding="utf-8")

# log
titanic_test_df.head()

## <a id='toc2_2_'></a>[Sales](#toc0_)

- split into relational dataframes:
  - orders $\leftrightarrow$ order_details $\rightarrow$ via ORDERNUMBER
  - order_details $\leftrightarrow$ products $\rightarrow$ via PRODUCTCODE
  - orders $\leftrightarrow$ customers $\rightarrow$ via CUSTOMERNAME (not ideal — real datasets use IDs, but fine for demonstration)


In [ ]:
# load sales
SALES_PATH = r"https://raw.githubusercontent.com/mr-pylin/datasets/refs/heads/main/data/tabular-data/sales/dataset.csv"
sales_df = pd.read_csv(SALES_PATH, encoding="latin1")

# log
sales_df.head()

In [ ]:
# split into relational dataframes
orders_df = sales_df[
    [
        "CUSTOMERNAME",
        "ORDERNUMBER",
        "ORDERDATE",
        "STATUS",
        "QTR_ID",
        "MONTH_ID",
        "YEAR_ID",
        "DEALSIZE",
    ]
].drop_duplicates()

order_details_df = sales_df[
    [
        "ORDERNUMBER",
        "ORDERLINENUMBER",
        "PRODUCTCODE",
        "QUANTITYORDERED",
        "PRICEEACH",
        "SALES",
    ]
].drop_duplicates()

products_df = sales_df[
    [
        "PRODUCTCODE",
        "PRODUCTLINE",
        "MSRP",
    ]
].drop_duplicates()

customers_df = sales_df[
    [
        "CUSTOMERNAME",
        "PHONE",
        "ADDRESSLINE1",
        "ADDRESSLINE2",
        "CITY",
        "STATE",
        "POSTALCODE",
        "COUNTRY",
        "TERRITORY",
        "CONTACTLASTNAME",
        "CONTACTFIRSTNAME",
    ]
].drop_duplicates()

# <a id='toc3_'></a>[Merging, Joining, and Reshaping](#toc0_)


## <a id='toc3_1_'></a>[Introduction to Data Combination](#toc0_)


### <a id='toc3_1_1_'></a>[Why merging and reshaping matter](#toc0_)

**Merging** and **reshaping** are essential operations in data manipulation that allow you to work with data in a more flexible and meaningful way.
- Often, real-world datasets are fragmented across multiple sources or tables, and you need to **merge** or **join** them to create a comprehensive view of the data.
- This helps to **combine related information** that was originally stored separately.
- **Reshaping** is equally important for reorganizing data into formats that better suit analysis or visualization.
- For instance, data may need to be converted from **long to wide format** (or vice versa) depending on the type of analysis or plotting needed.

✍️ **Key Use Cases**:
  - **Data Integration**: Merging data from different sources, such as customer information from one table and transaction details from another, to create a unified dataset.
  - **Data Transformation**: Reshaping data for analysis, such as summarizing transactions over time or converting categorical columns into multiple binary columns.

📝 **Docs**:

- `pandas.merge`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)
- `pandas.pivot`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html)
- `pandas.concat`: [pandas.pydata.org/docs/reference/api/pandas.concat.html](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)


### <a id='toc3_1_2_'></a>[Comparison of concatenation, merging, and reshaping](#toc0_)

**Concatenation**:
- **Concatenation** is the process of **stacking** two or more DataFrames along a particular axis (either rows or columns).
- It’s useful when you want to combine datasets that are similar in structure but different in content, such as appending new data to an existing dataset.

**merging**:
- **Merging** allows you to **combine DataFrames based on common columns or indexes**.
- It’s similar to SQL joins, where you can perform **inner**, **outer**, **left**, or **right** joins to bring together data from different tables based on matching keys.

**reshaping**:
- **Reshaping** involves changing the **structure** of your data without changing its content.
- It’s useful for converting data between different formats, such as transforming data from long to wide format or creating a pivot table for aggregation.

✍️ **Key Differences**:
- **Concatenation** is mostly used when you want to **stack** DataFrames vertically or horizontally.
- **Merging** is best when you need to combine DataFrames based on **matching key columns** or indexes, similar to database joins.
- **Reshaping** is for changing the **layout** of the data, such as summarizing or pivoting data for better analysis or visualization.

📝 **Docs**:

- `pandas.concat`: [pandas.pydata.org/docs/reference/api/pandas.concat.html](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
- `pandas.merge`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)
- `pandas.pivot_table`: [pandas.pydata.org/docs/reference/api/pandas.pivot_table.html](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)


### <a id='toc3_1_3_'></a>[Overview of relational data concepts](#toc0_)

- **Relational data** is organized into tables (also called **relations**). Each table has rows and columns.
- Relationships between tables are based on **common keys** or **attributes**.
- This concept is fundamental in databases and widely used in data science for combining datasets.
- In **relational data**, tables are linked using **primary keys** and **foreign keys**.
  - Primary keys uniquely identify each record in a table.
  - Foreign keys reference primary keys in other tables.
  - This structure helps reduce redundancy and supports **data normalization**.

**Relational concepts** in pandas include:
- **Merging**: Combine tables based on common columns or indexes (like SQL joins).  
- **Concatenation**: Stack multiple tables together. Useful for appending rows or columns.  
- **Pivoting and reshaping**: Change the structure of the data while preserving relationships.

**Key Relational Data Operations**:
- **JOINs**: Link tables based on matching keys, such as **inner** or **outer** joins.  
- **Aggregation**: Group data by attributes (e.g., sum of sales per customer).  
- **Normalization**: Organize data into separate tables to minimize duplication.

📝 **Docs**:

- `pandas.merge`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)


## <a id='toc3_2_'></a>[Concatenation and Appending](#toc0_)


### <a id='toc3_2_1_'></a>[Using `pd.concat()` for stacking DataFrames](#toc0_)

- The `pd.concat()` function in pandas allows you to **stack DataFrames** vertically or horizontally.
- This is useful when you need to **combine datasets** with the same structure but different content (e.g., appending rows or adding columns).

<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/images/pandas/tutorials/08/concat_row.svg" alt="Concatenating DataFrames" style="min-width: 256px; max-height: 40%; width: auto; background-color: #DBDBDB; border-radius: 16px;">
    <p><em>Figure 1: Combine data from multiple tables (Concatenating objects)</em> (<a href="https://pandas.pydata.org/docs/getting_started/intro_tutorials/" target="_blank">source</a>)</p>
</div>

**Key Parameters**:
- **axis**: Determines whether to concatenate along rows (`axis=0`) or columns (`axis=1`).
- **ignore_index**: Resets the index after concatenation, which is useful if the original indexes are not meaningful.
- **keys**: Adds a hierarchical index for concatenated data, making it easier to identify different data sources.
- **join**: Specifies how to handle mismatched columns, either by taking the union (`'outer'`) or intersection (`'inner'`) of columns.

**Use Cases**:
- **Vertical concatenation**: Useful when stacking rows from different DataFrames, such as when appending new records to an existing dataset.
- **Horizontal concatenation**: Useful for adding columns to a DataFrame, such as combining features from different data sources.

📝 **Docs**:

- `pandas.concat`: [pandas.pydata.org/docs/reference/api/pandas.concat.html](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)


In [ ]:
# concatenate vertically [row-wise]
vertical_concat = pd.concat([titanic_train_df, titanic_test_df], axis=0, ignore_index=True)

# log
print(f"titanic_train_df.shape : {titanic_train_df.shape}")
print(f"titanic_test_df.shape  : {titanic_test_df.shape}")
print(f"vertical_concat.shape  : {vertical_concat.shape}")

In [ ]:
titanic_sub_1 = titanic_train_df[["PassengerId", "Survived", "Pclass", "Name", "Sex", "Age"]]
titanic_sub_2 = titanic_train_df[["SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked"]]

# concatenate horizontally [column-wise]
horizontal_concat = pd.concat([titanic_sub_1, titanic_sub_2], axis=1)

# log
print(f"titanic_sub_1.shape : {titanic_sub_1.shape}")
print(f"titanic_sub_2.shape : {titanic_sub_2.shape}")
print(f"horizontal_concat.shape : {horizontal_concat.shape}")

### <a id='toc3_2_2_'></a>[Ignoring indexes and adding keys](#toc0_)

**Ignoring indexes**:
- When concatenating DataFrames, you may want to **ignore the original indexes**.
- This is useful when the existing indexes are not meaningful or differ between DataFrames.
- Using `ignore_index=True` in `pd.concat()` resets the index.
- The new DataFrame gets a fresh index starting from `0`.
- This is helpful when stacking DataFrames with different index labels.

**Adding keys**:
- The `keys` parameter allows you to create a **hierarchical index**.
- It labels the source of each DataFrame in the concatenation. This makes it easier to track where each row came from.

**Use Cases**:
- **Ignoring indexes**: Remove the original indexes and create a continuous new index.  
- **Adding keys**: Label each DataFrame in a multi-level index to track data origins.

📝 **Docs**:

- `pandas.concat`: [pandas.pydata.org/docs/reference/api/pandas.concat.html](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)


In [ ]:
ignored_index_concat = pd.concat([titanic_train_df, titanic_test_df], axis=0, ignore_index=True)

# log
ignored_index_concat.iloc[889:893]

In [ ]:
# using keys to create hierarchical index
# ignore_index=True overrides keys, so you should choose one
keyed_concat = pd.concat([titanic_train_df, titanic_test_df], axis=0, keys=["train", "test"])

# log
print(keyed_concat.loc["train"].head(), end="\n\n")
print(keyed_concat.loc["test"].head(), end="\n\n")
print(keyed_concat.loc["train", 0])

## <a id='toc3_3_'></a>[Merging and Joining](#toc0_)


### <a id='toc3_3_1_'></a>[The `merge()` function and SQL-style joins](#toc0_)

- The `merge()` function in pandas allows you to combine two DataFrames based on a common column or index, similar to **SQL joins**.
- It’s one of the most powerful tools for merging datasets in a flexible and customizable way.

<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/images/pandas/tutorials/08/merge_left.svg" alt="Join Tables" style="min-width: 256px; max-height: 40%; width: auto; background-color: #DBDBDB; border-radius: 16px;">
    <p><em>Figure 2: Join tables using a common identifier</em> (<a href="https://pandas.pydata.org/docs/getting_started/intro_tutorials/" target="_blank">source</a>)</p>
</div>

**Use Cases**:
- Combining customer information with their orders by matching on a customer ID.
- Merging sales data with product details to get a full view of the sales performance.

📝 **Docs**:

- `pandas.merge`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)


#### <a id='toc3_3_1_1_'></a>[Inner Join](#toc0_)

- Returns only the rows with **matching values** in both DataFrames. Rows without a match are excluded.
- It’s the most common join used when you want to retain only the overlapping data between two tables.


In [ ]:
# merge orders with customers on CUSTOMERNAME
orders_customers_inner = pd.merge(orders_df, customers_df, on="CUSTOMERNAME", how="inner")

# log
orders_customers_inner.head()

#### <a id='toc3_3_1_2_'></a>[Outer join](#toc0_)

- Returns all rows from both DataFrames, filling missing values with `NaN` where there is no match.
- Useful when you want to preserve all data from both DataFrames, even if some rows do not have corresponding matches.


In [ ]:
# merge orders and order_details to get all records, including unmatched ones
orders_orderdetails_outer = pd.merge(orders_df, order_details_df, on="ORDERNUMBER", how="outer")

# log
orders_orderdetails_outer.head()

#### <a id='toc3_3_1_3_'></a>[Left join](#toc0_)

- Returns all rows from the **left DataFrame** and only the rows from the **right DataFrame** where there are matches.
- This is useful when you want to retain all data from the left table and only add matching rows from the right.


In [ ]:
# merge order_details with products to get full order details
order_product_left = pd.merge(order_details_df, products_df, on="PRODUCTCODE", how="left")

# log
order_product_left.head()

#### <a id='toc3_3_1_4_'></a>[Right join](#toc0_)

- Returns all rows from the **right DataFrame** and only the rows from the **left DataFrame** where there are matches.
- This join is useful when you want to keep all data from the right table and add the corresponding data from the left.


In [ ]:
# merge customers with orders to include all customers even if they have no orders
customers_orders_right = pd.merge(customers_df, orders_df, on="CUSTOMERNAME", how="right")

# log
customers_orders_right.head()

### <a id='toc3_3_2_'></a>[Joining on indexes and multi-indexes](#toc0_)

- In pandas, you can join DataFrames using **indexes** or **multi-indexes** rather than just columns.
- This is particularly useful when the key for merging is the index or a combination of multiple indexes.

**Joining on Indexes**:
- You can join two DataFrames on their index using the `left_index=True` and `right_index=True` parameters in the `merge()` function.
- This approach is useful when the indexes themselves represent unique identifiers or categories, such as timestamps or product IDs.
  
**Multi-Indexes**:
- pandas allows for **hierarchical indexing**, where an index can have multiple levels (multi-indexes).
- You can join DataFrames on these multi-level indexes, which can represent more complex relationships between data.
- To merge on a multi-index, you can specify the `left_index` and `right_index` parameters and pandas will automatically use the multi-index for merging.

**Use Cases**:
- **Index-based joins**: Merging two DataFrames where one DataFrame has a date-time index and the other has a standard integer index, with the dates being the key for the join.
- **Multi-index joins**: Combining product sales data with product information, where each DataFrame uses a multi-index (e.g., region and product category).

📝 **Docs**:

- `pandas.merge`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)


In [ ]:
# joining on indexes
# set ORDERNUMBER as the index in both orders_df and order_details_df [join key]
orders_indexed = orders_df.set_index("ORDERNUMBER")
order_details_indexed = order_details_df.set_index("ORDERNUMBER")

# merge using indexes
merged_on_index = pd.merge(orders_indexed, order_details_indexed, left_index=True, right_index=True, how="inner")

# log
merged_on_index.head()

### <a id='toc3_3_3_'></a>[Handling overlapping column names (`suffixes` parameter)](#toc0_)

- When merging DataFrames, if both DataFrames have columns with the same name (besides the join keys), pandas will append a suffix to distinguish between them.
- This is controlled through the `suffixes` parameter in the `merge()` function.
- By default, pandas uses the suffixes `"_x"` and `"_y"` to differentiate the overlapping columns in the left and right DataFrames, respectively.
- You can customize these suffixes to better match your data's naming conventions.

**Use Cases**:
- When both DataFrames have a column with the same name (e.g., `value`), the `suffixes` parameter adds distinct labels to each column (like `value_x` and `value_y`).
- Custom suffixes help avoid confusion when overlapping columns represent different concepts in the merged DataFrame.

📝 **Docs**:

- `pandas.merge`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)


In [ ]:
# take small subsets for demonstration
orders_subset = orders_df[["ORDERNUMBER", "STATUS"]].head(3)
order_details_subset = order_details_df[["ORDERNUMBER", "SALES"]].head(3)

# introduce an overlapping column intentionally for demonstration
orders_subset["SALES"] = [1000, 1500, 1200]

# log
print(orders_subset.columns)
print(order_details_subset.columns)

In [ ]:
# merge without custom suffixes (default "_x" and "_y")
merged_default_suffix = pd.merge(orders_subset, order_details_subset, on="ORDERNUMBER", how="outer")

# log
merged_default_suffix

In [ ]:
# merge with custom suffixes
merged_custom_suffix = pd.merge(
    orders_subset, order_details_subset, on="ORDERNUMBER", how="outer", suffixes=("_orders", "_updates")
)

# log
merged_custom_suffix

## <a id='toc3_4_'></a>[Pivoting and Reshaping Data](#toc0_)


### <a id='toc3_4_1_'></a>[Converting long to wide format with `pivot()`](#toc0_)

- The `pivot()` function in pandas allows you to **reshape data** from a **long format** to a **wide format**.
- This is particularly useful when you need to **spread out data** that is stored in a single column across multiple columns.
- **Long format** refers to datasets where each row represents a single observation and each variable is stored in its own column.
- **Wide format** refers to datasets where each variable is spread across multiple columns, making it easier to compare categories side by side.

<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/images/pandas/tutorials/07/pivot.svg" alt="Long to wide table format" style="min-width: 256px; max-height: 40%; width: auto; background-color: #DBDBDB; border-radius: 16px;">
    <p><em>Figure 3: Long to wide table format using <code>pivot</code></em> (<a href="https://pandas.pydata.org/docs/getting_started/intro_tutorials/" target="_blank">source</a>)</p>
</div>
<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/images/pandas/tutorials/07/pivot_table.svg" alt="Pivot Table" style="min-width: 256px; max-height: 40%; width: auto; background-color: #DBDBDB; border-radius: 16px;">
    <p><em>Figure 4: Pivot Table</em> (<a href="https://pandas.pydata.org/docs/getting_started/intro_tutorials/" target="_blank">source</a>)</p>
</div>

**Use Cases**:
- **Pivoting sales data**: Converting a long-format sales dataset with individual transactions into a wide format where each product category is a separate column.
- **Reorganizing time series data**: Converting data that stores daily values in a single column into a wide format with one column for each day.

📝 **Docs**:

- `pandas.pivot`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html)


In [ ]:
# take a subset of order_details_df for demonstration
order_details_subset = order_details_df[["ORDERNUMBER", "PRODUCTCODE", "SALES"]].iloc[12:31]

# pivot: make PRODUCTCODE the columns, ORDERNUMBER the index, SALES as values
wide_format = order_details_subset.pivot(index="ORDERNUMBER", columns="PRODUCTCODE", values="SALES")

# log
wide_format

### <a id='toc3_4_2_'></a>[Aggregating with `pivot_table()`](#toc0_)

- The `pivot_table()` function in pandas is a more powerful version of `pivot()`.
- `pivot_table()` can compute various aggregations, such as the **sum** of sales or the **mean** of temperatures, depending on the values in your columns.
- It allows you to **aggregate data** while reshaping it from long format to wide format, making it ideal for summarizing large datasets.

**Use Cases**:
- **Sales Summaries**: You can aggregate sales data by product category and region, showing the total sales per region for each product category.
- **Time Series**: Aggregating daily stock prices by week or month to analyze trends.

📝 **Docs**:

- `pandas.pivot_table`: [pandas.pydata.org/docs/reference/api/pandas.pivot_table.html](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)


In [ ]:
# take a subset of order_details_df and orders_df for demonstration
order_details_subset = order_details_df[["ORDERNUMBER", "PRODUCTCODE", "SALES"]].iloc[12:31]
orders_subset = orders_df[["ORDERNUMBER", "DEALSIZE"]].iloc[12:31]

# merge to include DEALSIZE for aggregation
sales_data = pd.merge(order_details_subset, orders_subset, on="ORDERNUMBER", how="left")

# log
sales_data

In [ ]:
# pivot table: aggregate total sales by DEALSIZE (rows) and PRODUCTCODE (columns)
pivot_table_sales = pd.pivot_table(
    sales_data,
    index="DEALSIZE",       # rows
    columns="PRODUCTCODE",  # columns
    values="SALES",         # values to aggregate
    aggfunc="sum",          # aggregation function
    fill_value=0,           # replace missing combinations with 0
)

# log
pivot_table_sales

## <a id='toc3_5_'></a>[Melting and Long-Form Transformations](#toc0_)


### <a id='toc3_5_1_'></a>[Using `melt()` to unpivot data](#toc0_)

- The `melt()` function in pandas is used to **unpivot** data from a wide format to a long format.
- It takes columns and **converts them into rows**, which is often needed when preparing data for analysis or visualization.
- **Wide format** refers to data where each variable is stored in its own column
- **Long format** refers to data where variables are stored in a single column, and a separate column is used for the variable names (such as product categories or months).

<div style="text-align: center; padding-top: 10px;">
    <img src="../assets/images/pandas/tutorials/07/melt.svg" alt="Wide to long format" style="min-width: 256px; max-height: 40%; width: auto; background-color: #DBDBDB; border-radius: 16px;">
    <p><em>Figure 5: Wide to long format using <code>melt</code></em> (<a href="https://pandas.pydata.org/docs/getting_started/intro_tutorials/" target="_blank">source</a>)</p>
</div>

**Use Cases**:
- **Unpivoting a time series**: If you have columns representing each month of the year, you can use `melt()` to convert each month into a row, creating a long-format dataset for analysis.
- **Flattening a wide table**: Converting a DataFrame with multiple columns representing different measurements (e.g., height, weight, age) into a long format for statistical modeling.

**Key Parameters**:
- `id_vars`: The columns you want to keep fixed (these will not be melted).
- `value_vars`: The columns to unpivot (these will be converted into rows).
- `var_name`: The name to give to the new column that will hold the variable names (e.g., "Month").
- `value_name`: The name to give to the new column that will hold the values.

📝 **Docs**:

- `pandas.melt`: [pandas.pydata.org/docs/reference/api/pandas.melt.html](https://pandas.pydata.org/docs/reference/api/pandas.melt.html)


In [ ]:
# take a subset of order_details_df and orders_df for demonstration
order_details_subset = order_details_df[["ORDERNUMBER", "PRODUCTCODE", "SALES"]].head(6)
orders_subset = orders_df[["ORDERNUMBER", "DEALSIZE"]].head(6)

# merge to include DEALSIZE as an identifier
sales_data = pd.merge(order_details_subset, orders_subset, on="ORDERNUMBER", how="left")

# Log
sales_data

In [ ]:
# melt: unpivot PRODUCTCODE and SALES into long format
melted_sales = pd.melt(
    sales_data,
    id_vars=["ORDERNUMBER", "DEALSIZE"],  # columns to keep fixed
    value_vars=["PRODUCTCODE", "SALES"],  # columns to unpivot
    var_name="Variable",                  # name for the variable column
    value_name="Value",                   # name for the value column
)

# log
melted_sales

### <a id='toc3_5_2_'></a>[Reversing a melt operation](#toc0_)

- After using `melt()` to convert data from wide to long format, you may sometimes need to reverse the process and **pivot** the data back to its original wide format.
- This is useful when you want to reassemble the unpivoted data after performing analysis or transformations in long format.
- **Reversing a melt operation** can be done using the `pivot()` or `pivot_table()` functions.
- By specifying the correct **id variables** and **value variables**, you can reshape the data back to its wide form.

**Use Cases**:
- After aggregating or analyzing the long-form data (e.g., calculating the total sales per month), you may need to revert to the wide format to compare the results across different variables side by side.
- When reshaping data for a machine learning model and then needing to convert it back to its original format for reporting or visualization.

**Key Considerations**:
- To reverse a melt, make sure that the combination of **id variables** is unique after the melt operation, as `pivot()` will require unique index-variable pairs.
- If there were duplicates in the melted data, `pivot_table()` can be used with aggregation functions to handle them.

📝 **Docs**:

- `pandas.pivot`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html)
- `pandas.pivot_table`: [pandas.pydata.org/docs/reference/api/pandas.pivot_table.html](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)


In [ ]:
# reverse the melt using pivot() (long to wide)
reversed_sales = melted_sales.pivot(
    index=["ORDERNUMBER", "DEALSIZE"],  # id_vars used as index
    columns="Variable",                 # variable column becomes wide columns
    values="Value",                     # values column fills the data
).reset_index()

# log
reversed_sales

## <a id='toc3_6_'></a>[Stacking and Unstacking](#toc0_)


### <a id='toc3_6_1_'></a>[Using `stack()` and `unstack()` to reshape multi-indexed data](#toc0_)

- The `stack()` and `unstack()` functions in pandas are used to reshape **multi-indexed data**.
- These operations allow you to pivot the inner levels of the index, converting rows to columns and vice versa, which is useful for manipulating hierarchical data.
- Both `stack()` and `unstack()` work with multi-index DataFrames, so your DataFrame must have multiple levels of index or columns to apply these operations.

📝 **Docs**:

- `pandas.DataFrame.stack`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html)
- `pandas.DataFrame.unstack`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.unstack.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.unstack.html)


In [ ]:
# take a subset of sales data
sales_subset = pd.merge(
    order_details_df[["ORDERNUMBER", "PRODUCTCODE", "SALES"]].head(6),
    orders_df[["ORDERNUMBER", "DEALSIZE"]].head(6),
    on="ORDERNUMBER",
    how="left",
)

# log
sales_subset

#### <a id='toc3_6_1_1_'></a>[Stack](#toc0_)

- Converts **columns to rows** by "stacking" the inner levels of a multi-indexed DataFrame into a single column.
- This operation compresses the DataFrame by turning a **wide format** into a **long format**.
- It is often used when you need to collapse multiple columns into a single column and move the information to a more compact form.


In [ ]:
# create a hierarchical index (multi-index)
sales_multi_index = sales_subset.set_index(['DEALSIZE', 'ORDERNUMBER'])

# log
sales_multi_index

In [ ]:
# stack the DataFrame (columns → rows)
stacked_sales = sales_multi_index.stack()

# log
stacked_sales

#### <a id='toc3_6_1_2_'></a>[Unstack](#toc0_)

- The opposite of `stack()`, it converts **rows to columns** by "unstacking" the inner level of the index.
- This operation is useful when you need to spread out a long-format DataFrame into a wide format, turning each inner index level into a separate column.


In [ ]:
# unstack the DataFrame (rows → columns)
# by default, unstack moves the innermost index level to columns
unstacked_sales = stacked_sales.unstack()

# log
unstacked_sales

In [ ]:
# unstack a specific index level (e.g., DEALSIZE)
unstacked_level_sales = stacked_sales.unstack(level=0)

# log
unstacked_level_sales

### <a id='toc3_6_2_'></a>[Combining reshaping with aggregation for advanced layouts](#toc0_)

- Combining **reshaping operations** with **aggregation** functions allows you to create complex, **advanced layouts** that summarize and organize your data for deeper analysis.
- This approach is particularly useful when dealing with large datasets and when you need to extract meaningful insights from multiple variables.
- **Reshaping** (using functions like `pivot()`, `pivot_table()`, `stack()`, and `unstack()`) is often combined with **aggregation** (such as `sum()`, `mean()`, or `count()`) to produce summary tables or reports that represent the data in a more interpretable format.

**Use Cases**:
- **Sales analysis**: After reshaping data to show sales by region and product category (using `pivot_table()`), you can aggregate the results to compute the total sales, average sales, or number of sales transactions for each region and product category.
- **Time series analysis**: You can combine reshaped time series data with aggregation functions to compute statistics like moving averages or cumulative totals over time.

**Key Considerations**:
- **Aggregation after reshaping**: When performing reshaping, consider which aggregation function is appropriate for your analysis. For example, when analyzing sales data, you may want to aggregate by **sum** to get total sales, or by **mean** to get average sales per category.
- **Advanced layouts**: After reshaping and aggregating, you can create hierarchical tables with multi-level indexes to represent complex relationships between variables.

📝 **Docs**:

- `pandas.pivot_table`: [pandas.pydata.org/docs/reference/api/pandas.pivot_table.html](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)
- `pandas.DataFrame.stack`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html)
- `pandas.DataFrame.unstack`: [pandas.pydata.org/docs/reference/api/pandas.DataFrame.unstack.html](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.unstack.html)


In [ ]:
# merge product, order, and order details data for richer context
merged_sales = (
    order_details_df
    .merge(products_df[['PRODUCTCODE', 'PRODUCTLINE']], on='PRODUCTCODE', how='left')
    .merge(orders_df[['ORDERNUMBER', 'YEAR_ID', 'DEALSIZE']], on='ORDERNUMBER', how='left')
)

# log
merged_sales

In [ ]:
# create a pivot table with aggregation
# summarize total sales by PRODUCTLINE and DEALSIZE for each YEAR
sales_summary = pd.pivot_table(
    merged_sales,
    values='SALES',
    index=['YEAR_ID', 'PRODUCTLINE'],
    columns='DEALSIZE',
    aggfunc='sum',
    fill_value=0
)

# log
sales_summary

In [ ]:
# reshape the aggregated table using stack/unstack
# stack converts columns (DEALSIZE) into rows
stacked_sales = sales_summary.stack()

# log
stacked_sales.head()

In [ ]:
# unstack again by PRODUCTLINE to compare across deal sizes
unstacked_sales = stacked_sales.unstack(level='PRODUCTLINE')

# log
unstacked_sales.head()

In [ ]:
# optional - further aggregation
# Example: Average total sales per deal size across all product lines
avg_sales_per_dealsize = sales_summary.mean(axis=0)

# log
avg_sales_per_dealsize